In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset
import torch

# ── 1. 4-bit quantization config (the "Q" in QLoRA) ──────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,              # quantize weights to 4-bit
    bnb_4bit_quant_type="nf4",      # NormalFloat4 — best for LLMs
    bnb_4bit_compute_dtype=torch.float32,  # float32 for CPU (use bfloat16 on GPU)
    bnb_4bit_use_double_quant=True, # nested quantization = extra memory savings
)

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# ── 2. Load model in 4-bit ────────────────────────────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",  # auto-selects CPU or GPU
)

# ── 3. Prepare model for k-bit training (freezes quantized layers) ────────────
model = prepare_model_for_kbit_training(model)

# ── 4. Attach LoRA adapters on top of quantized model ────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["c_attn"],  # GPT-2 attention layer
    bias="none",                # QLoRA recommendation
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Output: trainable params: ~147K || all params: ~82M || trainable%: ~0.18%

# ── 5. Dataset ────────────────────────────────────────────────────────────────
data = [
    {"text": "Question: What is 2+2? Answer: 4"},
    {"text": "Question: Capital of France? Answer: Paris"},
    {"text": "Question: Who wrote Hamlet? Answer: Shakespeare"},
    {"text": "Question: What color is the sky? Answer: Blue"},
    {"text": "Question: What is the boiling point of water? Answer: 100°C"},
]
dataset = Dataset.from_list(data)

# ── 6. Training ───────────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./qlora-output",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,  # effective batch size = 4
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    optim="adamw_torch",  # use "paged_adamw_8bit" if on GPU for more savings
)

# SFTTrainer handles tokenization automatically
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    max_seq_length=64,
)

trainer.train()

# ── 7. Save ONLY the LoRA adapter weights (not the quantized base model) ──────
model.save_pretrained("./qlora-adapters")
tokenizer.save_pretrained("./qlora-adapters")
print("✅ Done! QLoRA adapters saved to ./qlora-adapters")

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [ ]:
# !pip install trl
# !pip install -U bitsandbytes>=0.46.1

zsh:1: 0.46.1 not found
